# Bonsai Ternary Training — Colab Free Tier

Train a 0.6B ternary model on a free T4 GPU.

**Setup:**
- Runtime → Change runtime type → T4 GPU
- Run cells top to bottom
- Checkpoint saves to HuggingFace Hub every 500 steps
- If session dies, just re-run — it resumes from last checkpoint

## 1. Check GPU

In [ ]:
!nvidia-smi
import torch
print(f"\nGPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
print(f"PyTorch: {torch.__version__}")
if torch.cuda.get_device_capability(0)[0] >= 9:
    print("FP8 supported (Hopper+)")
else:
    print(f"FP8 NOT supported (compute capability {torch.cuda.get_device_capability(0)})")
    print("Will use FP32 BitLinear (still works, just slower)")

## 2. Install dependencies

This installs our training package + all deps. Takes ~2 minutes.

In [ ]:
import os

# Clone from GitHub
if not os.path.exists('bonsai-llama'):
    !git clone https://github.com/maxta85/bonsai-llama.git
    %cd bonsai-llama
else:
    %cd bonsai-llama
    !git pull

!pip install -q -e ".[train,dev]"
print("\n✓ bonsai-llama installed")

## 3. Set up HuggingFace Hub for checkpoints

Get a token from https://huggingface.co/settings/tokens
Create a free account if you don't have one.
Create a token with 'write' permission.

In [ ]:
from huggingface_hub import HfApi, login
import getpass

# Login to HF Hub
token = getpass.getpass("Enter HF token (or press Enter to skip): ")
if token:
    login(token=token)
    api = HfApi()
    user = api.whoami()
    print(f"\n✓ Logged in as: {user['name']}")
    
    # Create a repo for checkpoints (if it doesn't exist)
    REPO_ID = f"{user['name']}/bonsai-checkpoints"
    try:
        api.create_repo(repo_id=REPO_ID, repo_type="model", exist_ok=True)
        print(f"✓ Checkpoint repo: {REPO_ID}")
    except Exception as e:
        print(f"Repo setup: {e}")
else:
    print("Skipping HF Hub — checkpoints will save to local disk only")
    REPO_ID = None

## 4. Training configuration

Adjust these for your run. Defaults are for a quick test on T4.

In [ ]:
# === Training config ===
TEACHER = "Qwen/Qwen3-0.6B"       # Teacher model (BF16, frozen)
STUDENT = "Qwen/Qwen3-0.6B"       # Student base (will be ternarized)
MODE = "1.58b"                     # 1.58b = ternary, 1b = binary
DATASET = "wikitext"               # wikitext or other HF dataset
BATCH_SIZE = 4                     # T4 has 16GB, 0.6B fits at batch=4
SEQ_LEN = 1024
GRAD_ACCUM = 4                     # Effective batch = 4 * 4 = 16
MAX_STEPS = 500                    # Adjust for longer runs
LR = 3e-4
WARMUP = 50
SAVE_EVERY = 100                   # Save checkpoint every N steps
TOPK = 100                         # Top-k KL (memory efficient)

# === Memory optimizations ===
# T4 doesn't support FP8 (needs Hopper+), so we skip --fp8
# But we use all other optimizations
USE_8BIT_ADAM = True               # Saves 75% optimizer memory
USE_GRAD_CHECKPOINT = True         # Saves activation memory
USE_TOPK_KL = True                 # Saves ~14GB vocab logits
USE_CHUNKED_LOSS = True            # Avoids full vocab logits

print("Configuration:")
print(f"  Teacher:  {TEACHER}")
print(f"  Student:  {STUDENT} ({MODE})")
print(f"  Batch:    {BATCH_SIZE} × {GRAD_ACCUM} = {BATCH_SIZE*GRAD_ACCUM} effective")
print(f"  Steps:    {MAX_STEPS}")
print(f"  Memory:   8bit-adam={USE_8BIT_ADAM}, ckpt={USE_GRAD_CHECKPOINT}, "
          f"topk-kl={USE_TOPK_KL}, chunked={USE_CHUNKED_LOSS}")

## 5. Run training

This runs the actual training. Checkpoint saves to HF Hub every `SAVE_EVERY` steps.
If the session dies, just re-run this cell — it resumes from the last checkpoint.

In [ ]:
import os
import sys

# Build the training command
cmd = f"""
python -m training.distill \
  --teacher {TEACHER} \
  --student {STUDENT} \
  --mode {MODE} \
  --teacher-device cuda:0 \
  --student-device cuda:0 \
  --teacher-dtype bfloat16 \
  --dataset {DATASET} \
  --batch-size {BATCH_SIZE} \
  --seq-len {SEQ_LEN} \
  --grad-accum {GRAD_ACCUM} \
  --max-steps {MAX_STEPS} \
  --warmup-steps {WARMUP} \
  --lr {LR} \
  --log-every 10 \
  --save-every {SAVE_EVERY} \
  --out bonsai-checkpoint \
  --topk {TOPK}
"""

if USE_TOPK_KL:
    cmd += "  --topk-kl\n"
if USE_CHUNKED_LOSS:
    cmd += "  --chunked-loss\n"
if USE_8BIT_ADAM:
    cmd += "  --8bit-adam\n"
if USE_GRAD_CHECKPOINT:
    cmd += "  --grad-checkpoint\n"

print("Running:")
print(cmd)
print("-" * 60)

!{cmd}

## 6. Save checkpoint to HuggingFace Hub

Upload the trained model so it survives when the session ends.

In [ ]:
if REPO_ID and os.path.exists('bonsai-checkpoint'):
    from huggingface_hub import HfApi
    api = HfApi()
    
    print(f"Uploading checkpoint to {REPO_ID}...")
    api.upload_folder(
        folder_path="bonsai-checkpoint",
        repo_id=REPO_ID,
        repo_type="model",
    )
    print(f"✓ Uploaded to https://huggingface.co/{REPO_ID}")
else:
    print("No HF token or no checkpoint — skipping upload")
    print("Checkpoint is in ./bonsai-checkpoint (will be lost when session ends!)")

## 7. Test the model (optional)

Generate some text to see what the model learned.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained("bonsai-checkpoint")
model = AutoModelForCausalLM.from_pretrained("bonsai-checkpoint", torch_dtype=torch.float32).cuda()

prompt = "The capital of France is"
inputs = tok(prompt, return_tensors="pt").to("cuda:0")
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=50, do_sample=True, temperature=0.7)
print(tok.decode(out[0], skip_special_tokens=True))

## 8. Heartbeat (keep session alive)

Run this in a separate cell to prevent idle timeout (90 min).
This just prints every minute so Colab doesn't kill the session.

In [ ]:
import time
from datetime import datetime

print("Heartbeat running. Keep this tab open.")
print("This prevents Colab from killing the session for inactivity.")
print("Stop this cell when training is done.")
print()

for i in range(9999):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] alive ({i} min)", flush=True)
    time.sleep(60)